# Representation specificity and fixed-policy boundaries
Proofs: [theory_main.md](theory_main.md). The new affine controls distinguish coordinates from a changed ridge regularizer; they do not train HSE.

In [ ]:
import numpy as np
from experiments.p19.routing import weighted_headroom, conditional_regret
from experiments.p19.toy_routing import check_boundaries
from experiments.p19.affine_controls import fit_coordinates, ridge_fit, predict
check_boundaries()
np.testing.assert_allclose(weighted_headroom([[0,1],[1,0]],[.9,.1]),.1)
print('Hard headroom and fusion boundary passed')

In [ ]:
rng=np.random.default_rng(7)
x=rng.normal(size=(64,4))*[.1,1,3,10]
xt=rng.normal(size=(20,4))*[.1,1,3,10]
y=rng.normal(size=(64,2))
center,coordinates,eigenvalues=fit_coordinates(x)
w,b=ridge_fit(x,y,center,coordinates['R'],.1)
reference=predict(xt,center,coordinates['R'],w,b)
for name,a in coordinates.items():
    w,b=ridge_fit(x,y,center,a,.1,'matched')
    error=np.max(np.abs(predict(xt,center,a,w,b)-reference))
    assert error<1e-10
    print(name,'matched maximum error',float(error))
w,b=ridge_fit(x,y,center,coordinates['whitened_R'],.1,'isotropic')
changed=np.max(np.abs(predict(xt,center,coordinates['whitened_R'],w,b)-reference))
assert changed>.01
print('Unmatched whitening changes the objective:',float(changed))

In [ ]:
source=np.array([[.1,.3],[.3,.1]])
target=source[:,::-1]
regret=conditional_regret(target,np.argmin(source,axis=1),[.5,.5])
np.testing.assert_allclose(regret,.2)
assert regret<=2*np.max(abs(target-source))+1e-12
print('Reversal sensitivity, not an observable guarantee:',regret)

In [ ]:
r=np.array([-1.,0.,1.]);y=r*r
np.testing.assert_allclose(np.mean((y-y.mean())**2),2/9)
print('Nonlinear square target can help a restricted linear consumer.')
print('This is not evidence for the current AFFINE mean readout.')